In [0]:
# Purpose: Silver layer - type casting, normalization, dedup, validation checks
# Author: Erza Ademi - Data Quality & Rules Engineer (R5)
# Last updated: 2026-06-11
# Dependencies: bronze.metadata_raw_bronze

import dlt
from pyspark.sql import functions as F
from pyspark.sql.window import Window


@dlt.table(
    name="metadata_validated",
    comment="Silver: typed, normalized, deduplicated metadata with hard expectations"
)
@dlt.expect_or_drop("valid_column_id", "column_id IS NOT NULL")
@dlt.expect_or_drop("valid_column_name", "column_name IS NOT NULL")
@dlt.expect_or_drop("valid_table_id", "table_id IS NOT NULL")
def metadata_validated():

    df = dlt.read("dbx_metadata_governance_dev.bronze.metadata_raw_bronze")

    # ── VAL-01: Type casting ──
    df = df \
        .withColumn("pii_flag",
                    F.col("pii_flag").cast("boolean")) \
        .withColumn("critical_data_element_flag",
                    F.col("critical_data_element_flag").cast("boolean")) \
        .withColumn("total_record_count",
                    F.col("total_record_count").cast("bigint")) \
        .withColumn("invalid_record_count",
                    F.col("invalid_record_count").cast("bigint"))

    # ── VAL-02: Normalize — trim + empty→NULL ──
    string_cols = [c for c, t in df.dtypes if t == "string"]
    for c in string_cols:
        df = df.withColumn(
            c, F.when(F.trim(F.col(c)) == "", None).otherwise(F.trim(F.col(c)))
        )

    # Conf → Confidential + title-case certification
    df = df \
        .withColumn("security_classification",
            F.when(F.col("security_classification") == "Conf", "Confidential")
             .otherwise(F.col("security_classification"))) \
        .withColumn("certification_level",
            F.initcap(F.col("certification_level")))

    # ── VAL-03: Dedup on column_id ──
    w = Window.partitionBy("column_id").orderBy(F.col("column_id"))
    df = df.withColumn("_rn", F.row_number().over(w)) \
           .filter(F.col("_rn") == 1) \
           .drop("_rn")

    # ── VAL-05: 11 per-row checks ──
    df = df \
        .withColumn("has_column_desc",
                    F.col("column_desc").isNotNull()) \
        .withColumn("has_term_name",
                    F.col("term_name").isNotNull()) \
        .withColumn("has_term_description",
                    F.col("term_description").isNotNull()) \
        .withColumn("has_security_classification",
                    F.col("security_classification").isNotNull()) \
        .withColumn("has_data_steward",
                    F.col("data_steward").isNotNull()) \
        .withColumn("has_table_desc",
                    F.col("table_desc").isNotNull()) \
        .withColumn("has_tag",
                    F.col("tag_value").isNotNull()) \
        .withColumn("has_subdomain",
                    F.col("term_subdomain").isNotNull()) \
        .withColumn("has_location",
                    F.col("location").isNotNull()) \
        .withColumn("pii_consistency",
            (F.col("pii_flag") == False) |
            (F.col("security_classification") == "Confidential")
        ) \
        .withColumn("dq_pass",
            (F.col("total_record_count") > 0) &
            (F.col("invalid_record_count") / F.col("total_record_count") < 0.10)
        )

    # ── VAL-06: dq_invalid_pct + completeness_pct + certification level ──
    checks = [
        "has_column_desc", "has_term_name", "has_term_description",
        "has_security_classification", "has_data_steward", "has_table_desc",
        "has_tag", "has_subdomain", "has_location", "pii_consistency", "dq_pass"
    ]

    df = df \
        .withColumn("dq_invalid_pct",
            F.when(F.col("total_record_count") > 0,
                F.col("invalid_record_count") / F.col("total_record_count") * 100
            ).otherwise(None)
        ) \
        .withColumn("completeness_pct",
            sum([F.col(c).cast("int") for c in checks]) / 11 * 100
        ) \
        .withColumn("achieved_cert_level",
            F.when(
                F.col("has_security_classification") &
                F.col("has_column_desc") &
                F.col("has_table_desc") &
                F.col("has_term_name") &
                F.col("has_data_steward") &
                F.col("has_term_description") &
                F.col("has_tag") &
                F.col("has_subdomain") &
                F.col("pii_consistency") &
                F.col("dq_pass"), "Certified"
            ).when(
                F.col("has_security_classification") &
                F.col("has_column_desc") &
                F.col("has_table_desc") &
                F.col("has_term_name") &
                F.col("has_data_steward"), "Documented"
            ).when(
                F.col("has_security_classification"), "Registered"
            ).otherwise("None")
        )

    return df